In [19]:
import matplotlib as mpl
import os
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

In [20]:
exp_root = "/media/yesindeed/DATADRIVE1/mount/remote_cse/experiments/med_vlm_benchmark/merged"

df_results = pd.read_csv(os.path.join(exp_root, "exp_status.csv"))
df_results = df_results.loc[~df_results["trainable_module"].isin(
    ["mdagent", "ucagent"])].reset_index(drop=True)
df_results["trainable_module"].fillna(value="ZS", inplace=True)

df_results

/tmp/ipykernel_2242976/3640445213.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_results["trainable_module"].fillna(value="ZS", inplace=True)


,model,task,dataset,model_type,trainable_module,path,have_eval_result,have_prediction,have_gpt_score,model_family,if_finished,error
0,Qwen2-VL,vqa,SLAKE,general,ZS,vqa/SLAKE/Qwen2-VL/eval_seed0/Qwen2-VL-7B-Inst...,1,1,1,Qwen,1,NaN
1,Qwen25-VL,vqa,SLAKE,general,ZS,vqa/SLAKE/Qwen25-VL/eval_seed0/Qwen2.5-VL-7B-I...,1,1,1,Qwen,1,NaN
2,Gemma3,vqa,SLAKE,general,ZS,vqa/SLAKE/Gemma3/eval_seed0/gemma-3-4b-it,1,1,1,Gemma,1,NaN
3,MedGemma,vqa,SLAKE,medical,ZS,vqa/SLAKE/MedGemma/eval_seed0/medgemma-4b-it,1,1,1,Gemma,1,NaN
4,InternVL3,vqa,SLAKE,general,ZS,vqa/SLAKE/InternVL3/eval_seed0/InternVL3-8B-hf,1,1,1,Intern,1,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
99,MedGemma,vqa,OmniMedVQA,medical,ML,vqa/OmniMedVQA/MedGemma/eval_seed0/train_lora_...,1,1,0,Gemma,1,NaN
100,InternVL3,vqa,OmniMedVQA,general,ML,vqa/OmniMedVQA/InternVL3/eval_seed0/intern-vl-...,1,1,0,Intern,1,NaN
101,LLaVA-1.5,vqa,OmniMedVQA,general,ML,vqa/OmniMedVQA/LLaVA-1.5/eval_seed0/train_lora...,1,1,0,LLaVA,1,NaN
102,LLaVA-Med,vqa,OmniMedVQA,general,ML,vqa/OmniMedVQA/LLaVA-Med/eval_seed0/train__M_s...,0,0,0,LLaVA,0,NaN


In [21]:
df_results.loc[df_results["if_finished"] == 0]

,model,task,dataset,model_type,trainable_module,path,have_eval_result,have_prediction,have_gpt_score,model_family,if_finished,error
45,gemini-2.5-pro,vqa,MedXpertQA,general,ZS,vqa/MedXpertQA/gemini-2.5-pro/eval_seed0/none,0,0,0,Gemini,0,NaN
46,o3,vqa,MedXpertQA,general,ZS,vqa/MedXpertQA/o3/eval_seed0/none,0,0,0,o-family,0,NaN
47,Lingshu,vqa,MedXpertQA,general,ZS,vqa/MedXpertQA/Lingshu/eval_seed0/Lingshu-7B,0,0,0,Qwen,0,NaN
55,gemini-2.5-pro,vqa,OmniMedVQA,medical,ZS,vqa/OmniMedVQA/gemini-2.5-pro/eval_seed0/none,0,0,0,Gemini,0,NaN
56,o3,vqa,OmniMedVQA,medical,ZS,vqa/OmniMedVQA/o3/eval_seed0/none,0,0,0,o-family,0,NaN
95,Lingshu,vqa,MedXpertQA,general,ML,vqa/MedXpertQA/Lingshu/eval_seed0/1epoch-lora8,0,0,0,Qwen,0,NaN
102,LLaVA-Med,vqa,OmniMedVQA,general,ML,vqa/OmniMedVQA/LLaVA-Med/eval_seed0/train__M_s...,0,0,0,LLaVA,0,NaN


In [22]:
df_results["trainable_module"].value_counts()

trainable_module
ZS    58
ML    46
Name: count, dtype: int64

In [23]:
import json
from tqdm import tqdm

acc = []
ci_low = []
ci_high = []

datasets_expect_closed_test_num = {
    "SLAKE": 416,
    "PathVQA": 3362,
    "VQA-RAD": 251,
    "Harvard-FairVLMed10k": 1996,
    "MedXpertQA": 400,
    "OmniMedVQA": 6186,
}

bootstrap_indices = {}

seed = 42
rng = np.random.default_rng(seed)
n_bootstraps = 1000

for i in tqdm(range(len(df_results))):
    row = df_results.iloc[i]
    dataset = row["dataset"]

    if row["if_finished"] != 1:
        mean_auc = lower = upper = np.nan

    else:
        with open(os.path.join(exp_root, row["path"], "bootstrap_closed_meta.json")) as fp:
            data = json.load(fp)

        acc_list = []

        for boots in data.values():
            acc_list.append(np.mean(boots["accuracy"]))

        acc_array = np.array(acc_list)

        mean_auc = np.mean(acc_array)
        lower = np.percentile(acc_array, 2.5)
        upper = np.percentile(acc_array, 97.5)

    acc.append(mean_auc)
    ci_low.append(lower)
    ci_high.append(upper)

df_results["acc"] = acc
df_results["ci_low"] = ci_low
df_results["ci_high"] = ci_high

df_results

100%|██████████| 104/104 [00:47<00:00,  2.18it/s]


,model,task,dataset,model_type,trainable_module,path,have_eval_result,have_prediction,have_gpt_score,model_family,if_finished,error,acc,ci_low,ci_high
0,Qwen2-VL,vqa,SLAKE,general,ZS,vqa/SLAKE/Qwen2-VL/eval_seed0/Qwen2-VL-7B-Inst...,1,1,1,Qwen,1,NaN,0.779962,0.737981,0.819712
1,Qwen25-VL,vqa,SLAKE,general,ZS,vqa/SLAKE/Qwen25-VL/eval_seed0/Qwen2.5-VL-7B-I...,1,1,1,Qwen,1,NaN,0.755822,0.713942,0.798077
2,Gemma3,vqa,SLAKE,general,ZS,vqa/SLAKE/Gemma3/eval_seed0/gemma-3-4b-it,1,1,1,Gemma,1,NaN,0.721779,0.680288,0.764423
3,MedGemma,vqa,SLAKE,medical,ZS,vqa/SLAKE/MedGemma/eval_seed0/medgemma-4b-it,1,1,1,Gemma,1,NaN,0.770599,0.730769,0.810096
4,InternVL3,vqa,SLAKE,general,ZS,vqa/SLAKE/InternVL3/eval_seed0/InternVL3-8B-hf,1,1,1,Intern,1,NaN,0.744175,0.701863,0.786058
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99,MedGemma,vqa,OmniMedVQA,medical,ML,vqa/OmniMedVQA/MedGemma/eval_seed0/train_lora_...,1,1,0,Gemma,1,NaN,0.929749,0.923052,0.936150
100,InternVL3,vqa,OmniMedVQA,general,ML,vqa/OmniMedVQA/InternVL3/eval_seed0/intern-vl-...,1,1,0,Intern,1,NaN,0.967738,0.962977,0.972195
101,LLaVA-1.5,vqa,OmniMedVQA,general,ML,vqa/OmniMedVQA/LLaVA-1.5/eval_seed0/train_lora...,1,1,0,LLaVA,1,NaN,0.903675,0.896056,0.910928
102,LLaVA-Med,vqa,OmniMedVQA,general,ML,vqa/OmniMedVQA/LLaVA-Med/eval_seed0/train__M_s...,0,0,0,LLaVA,0,NaN,NaN,NaN,NaN


In [24]:
df_results.to_csv("results/boot_results_vqa.csv", index=False)